# SupeRR Selector S2 - Round 1: Most reliable players under pressure
Step-by-step trace of the pipeline in `pressure/`. Run top to bottom. In Colab: upload the project folder (with `Data Pack/` and `pressure/`) to Drive and set `PROJECT_DIR`.

In [1]:
import os, sys, subprocess
PROJECT_DIR = os.environ.get('PROJECT_DIR', os.getcwd())  # Colab: '/content/drive/MyDrive/<folder>'
os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
# installs into whichever kernel is running (local or Colab)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import pandas as pd
from IPython.display import Image, display
from pressure import config as C
from pressure.data_prep import load_raw, prepare_matches, prepare_balls, validate
from pressure.win_prob import cross_fit, evaluate, temporal_holdout
from pressure.expected import add_expected
from pressure.pressure_index import add_pressure, agreement, component_correlations
from pressure.metrics import player_table, describe
from pressure.sensitivity import run_grid, one_at_a_time
pd.set_option('display.width', 200, 'display.max_columns', 40)

## 1. Data preparation & validation

In [2]:
raw = load_raw()
matches = prepare_matches(raw)
balls = prepare_balls(raw, matches)
report, mismatches = validate(balls, matches)
print(report)
mismatches

Balls (innings 1-2): 200,961
Matches with balls: 867
Matches with both innings and a recorded winner: 842
Winner reconstructed correctly (ties excluded): 99.76%
Tied (super-over decided): 9
Matches without winner (no result/abandoned/missing): 41
Legal balls per completed over == 6: 98.21%


,total_1,total_2,team_1,team_2,wkts_1,wkts_2,win_team_id,revised_target,result
match_id,,,,,,,,,
24822,139.0,47.0,23.0,18.0,7.0,2.0,18.0,NaN,India Won by 2 Runs (DLS method)
28334,110.0,95.0,21.0,17.0,10.0,5.0,17.0,NaN,New Zealand won by 17runs (DLS Method)


## 2. Win-probability model -> WPA and Leverage Index (out-of-fold)

In [3]:
balls, outcome_p = cross_fit(balls)
oof_metrics, calib = evaluate(balls)
hold_metrics, _ = temporal_holdout(balls)
display(pd.DataFrame({'out_of_fold': oof_metrics, 'holdout_2026': hold_metrics}).round(4))
calib

,out_of_fold,holdout_2026
balls,199029.0000,32631.0000
brier,0.1962,0.2109
brier_coinflip,0.2500,0.2500
log_loss,0.5886,0.6426
auc,0.7758,0.7570
brier_inn1,0.2255,0.2315
brier_inn2,0.1641,0.1881


,mean_pred,observed,n
"(-0.001, 0.1]",0.027064,0.085333,29426
"(0.1, 0.2]",0.150781,0.237671,12349
"(0.2, 0.3]",0.251599,0.326327,14320
"(0.3, 0.4]",0.352901,0.422580,17644
"(0.4, 0.5]",0.453258,0.451185,24777
"(0.5, 0.6]",0.547569,0.551144,28635
"(0.6, 0.7]",0.646524,0.630567,20342
"(0.7, 0.8]",0.747807,0.677393,14575
"(0.8, 0.9]",0.848590,0.729901,12003
"(0.9, 1.0]",0.974610,0.865133,24958


## 3. Expected runs / wickets and run value per ball

In [4]:
balls, wicket_value = add_expected(balls)
print('Runs a wicket costs, by phase:', wicket_value.round(2).to_dict())
balls.loc[balls.faced, ['runs', 'x_bat_runs', 'striker_out', 'x_bat_out']].mean()

Runs a wicket costs, by phase: {'death': 2.71, 'middle': 7.47, 'powerplay': 12.66}


runs           1.394590
x_bat_runs     1.395044
striker_out    0.054623
x_bat_out      0.054944
dtype: float64

## 4. Pressure definition

In [5]:
balls = add_pressure(balls, C.PRESSURE_PCTL)
display(pd.Series(agreement(balls)).round(3))
component_correlations(balls).round(3)

spearman_li_vs_rule           0.359
pressure_overlap_jaccard      0.239
share_pressure_balls_inn2     0.655
share_pressure_balls_death    0.240
dtype: float64

,innings_1,innings_2
wickets_down,-0.105,-0.158
death_overs,0.026,-0.082
recent_collapse,-0.056,-0.092
dot_streak,-0.020,-0.013
required_rate,NaN,-0.018
wp_closeness,0.280,0.865
rule_score,0.233,0.574


## 5-6. Player ratings and sensitivity

In [ ]:
results = {}
for role in ('bat', 'bowl'):
    tab, _ = player_table(balls, role)
    summ, R, _, taus = run_grid(balls, role)
    base = f"li|{C.PRESSURE_PCTL}|{C.MIN_PRESSURE_BALLS_BAT if role == 'bat' else C.MIN_PRESSURE_BALLS_BOWL}|full|base"
    display(one_at_a_time(taus, base).groupby('changed').kendall_tau.agg(['median', 'min']).round(3))
    tab = tab.join(summ[['median_rank', 'mean_rank', 'top10_rate']]).sort_values(['mean_rank', 'median_rank'])
    tab['final_rank'] = range(1, len(tab) + 1)
    results[role] = describe(tab, balls, role, raw['teams'])
    display(results[role][['final_rank', 'name', 'latest_team', 'p_balls', 'p_rv_raw', 'n_rv_raw', 'p_rv_eb',
                           'delta_rv', 'pos_share', 'mean_rank', 'top10_rate']].head(15).round(3))

## 7. Charts (generated by `python run_pipeline.py`)

In [ ]:
for f in ['02_wp_calibration.png', '04_leverage_heatmap.png', '04_chase_leverage_curve.png',
          '07_bat_pressure_vs_normal.png', '07_bat_top10.png', '07_bowl_pressure_vs_normal.png', '07_bowl_top10.png']:
    p = C.OUT_DIR / f
    if p.exists():
        display(Image(str(p)))